In [ ]:
import torch
from protlig.testing_helpers import GCNLayerSparse, load_dataset, mol_to_graph
import torch.nn as nn

In [5]:
def build_S(edge_index, num_nodes, dtype=torch.float32):
    """S = D~^-1/2 (A + I) D~^-1/2, built densely. Used only for checking."""
    A = torch.zeros(num_nodes, num_nodes, dtype=dtype)
    A[edge_index[0], edge_index[1]] = 1.0
    A = A + torch.eye(num_nodes, dtype=dtype)
    d = A.sum(dim=1).pow(-0.5)
    return d.unsqueeze(1) * A * d.unsqueeze(0)


def manual_grads(x, S, W, G):
    """Given upstream gradient G = dL/d(out), return dL/dW, dL/db, dL/dx."""
    dW = x.T @ S.T @ G            # (F_in, F_out)
    db = G.sum(dim=0)             # (F_out,)
    dx = S.T @ G @ W.T            # (N, F_in)
    return dW, db, dx

In [21]:
data = load_dataset()
smiles = data['SMILES']
ys = data['label']
smiles = smiles[0]
y = ys[0]

x, ei, y = mol_to_graph(smiles, y)
x = x.clone().requires_grad_(True)

layer = GCNLayerSparse(x.size(1), 10)

out = layer(x, ei)
target = torch.randn_like(out)
loss = ((out - target) ** 2).mean()
loss.backward()

# Upstream gradient of this loss, in closed form.
G = 2.0 * (out.detach() - target) / out.numel()

S = build_S(ei, x.shape[0])
dW, db, dx = manual_grads(x.detach(), S, layer.W.detach(), G)

print(float((dW - layer.W.grad).abs().max()))
print(float((db - layer.b.grad).abs().max()))
print(float((dx - x.grad).abs().max()))

7.450580596923828e-08
1.4901161193847656e-08
1.862645149230957e-09
